# Imágenes v2
Mapas espaciales de ROIs y vistas por temperatura. Usa las tablas de `fluo-ttl run` (`resultados/`) y las imágenes que dejan los macros de Fiji.

Por registro se buscan, dentro de su carpeta:
- `limpieza_ij/*_clean_view.tif` y `*_clean_quant.tif` (imagen alineada)
- `rois_ij/*_RoiSet.zip` (contornos) o `*_rois_table_ij.csv` (solo centroides)

Lo que falte se avisa y se omite. Requiere `pip install -e .[imagenes]` (tifffile, roifile).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, pandas as pd, matplotlib.pyplot as plt
from fluo_ttl import config, images

DATOS = '/Volumes/TOSHIBA EXT/Germán Fernandez/TRPM3/Fluo4'   # carpeta raíz usada en fluo-ttl run
RES   = 'resultados_mac'                                     # carpeta de salida de fluo-ttl run
cfg   = config.cargar()
reg   = pd.read_csv(f'{RES}/registros.csv', dtype={'genotipo': str})
por   = pd.read_csv(f'{RES}/por_roi.csv', dtype={'genotipo': str})
tab   = pd.read_csv(f'{RES}/tabla_genotipos.csv', dtype={'genotipo': str})
P99   = float(tab.P99_ref.iloc[0])
OUT   = f'{RES}/figuras_imagenes'; os.makedirs(OUT, exist_ok=True)
inc   = reg[reg.incluido]

# ¿qué hay disponible en cada registro?
disp = pd.DataFrame([{**{'genotipo': f.genotipo, 'registro': f.registro}, **{k: v is not None for k, v in images.archivos_registro(f, DATOS).items()}} for _, f in inc.iterrows()])
disp

## 1. Mapa espacial de un registro
Cada ROI coloreado por D (naranja = aumenta a T alta, violeta = disminuye). Contorno rojo = D por encima de P99 de Wt.

In [ ]:
GENO, REG = '1236', 17
f = inc[(inc.genotipo == GENO) & (inc.registro == REG)].iloc[0]
fig, ax = plt.subplots(figsize=(8, 8))
ax, motivo = images.mapa_espacial(f, DATOS, por, ax=ax, vmax=5, p99=P99)
print(motivo or 'ok'); fig.savefig(f'{OUT}/mapa_{GENO}_{REG}.png', dpi=130, bbox_inches='tight')

## 2. Mapas de todos los registros de un genotipo

In [ ]:
import numpy as np
GENO = '1236'
sub = inc[inc.genotipo == GENO]
nc = min(len(sub), 3); nr = int(np.ceil(len(sub) / nc))
fig, axs = plt.subplots(nr, nc, figsize=(6 * nc, 6 * nr), squeeze=False)
for a in axs.ravel(): a.axis('off')
for a, (_, f) in zip(axs.ravel(), sub.iterrows()):
    a.axis('on'); _, motivo = images.mapa_espacial(f, DATOS, por, ax=a, vmax=5, p99=P99)
    if motivo: a.text(.5, .5, motivo, ha='center', va='center', wrap=True, transform=a.transAxes, fontsize=8)
fig.tight_layout(); fig.savefig(f'{OUT}/mapas_{GENO}.png', dpi=110)

## 3. Vistas por temperatura
Media de los frames de cada temperatura (±0.5 °C) recortada alrededor de cada ROI. Cada fila tiene su propia escala de gris. Se muestran los ROI con mayor D del registro (`modo='baja'` para los que más disminuyen).

In [ ]:
GENO, REG = '1236', 17
f = inc[(inc.genotipo == GENO) & (inc.registro == REG)].iloc[0]
rois = images.top_rois(por, GENO, REG, n=6, modo='sube')
fig, motivo = images.vistas_por_temperatura(f, DATOS, cfg, rois, temps=(22, 26, 30, 34, 38), caja=40, ruta=f'{OUT}/temperatura_{GENO}_{REG}.png')
print(motivo or f'ROIs: {rois}')

## Notas
- El orden de los ROI debe coincidir entre el `RoiSet.zip` y las columnas de `ROI_Results.csv`; si el número no coincide con `por_roi.csv`, el mapa se omite con un aviso.
- Los ROI están en las coordenadas del stack alineado (`clean_quant`), no del original.
- Los frames de transición y los atípicos se excluyen de las medias por temperatura.
- Las medias de temperaturas con pocos frames (rampas rápidas) son más ruidosas; con menos de 3 frames la celda queda vacía.